<a href="https://colab.research.google.com/github/jfdelosrios/decision_tree_random_forest_project/blob/master/decision_tree_random_forest_project_template_fd59d141-d477-4f10-b39e-8127e0fbccb8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto práctico: árbol de decisión y random forest con scikit-learn

In [1]:
#Importamos las librerias principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Utilizaremos el **Car Evaluation Data Set** de Kaggle: https://www.kaggle.com/datasets/elikplim/car-evaluation-data-set

In [2]:
# https://archive.ics.uci.edu/dataset/19/car+evaluation

!pip install ucimlrepo

In [49]:
#Cargamos dataset a utilizar

from ucimlrepo import fetch_ucirepo

# fetch dataset
car_evaluation = fetch_ucirepo(id=19)

# data (as pandas dataframes)
X = car_evaluation.data.features.copy()
y = car_evaluation.data.targets.copy()

## Análisis exploratorio de datos

In [10]:
#Visualizacion del dataframe

print(X.head())
print(y.head())


  buying  maint doors persons lug_boot safety
0  vhigh  vhigh     2       2    small    low
1  vhigh  vhigh     2       2    small    med
2  vhigh  vhigh     2       2    small   high
3  vhigh  vhigh     2       2      med    low
4  vhigh  vhigh     2       2      med    med
   class
0  unacc
1  unacc
2  unacc
3  unacc
4  unacc


In [19]:
# variable information
car_evaluation.variables

,name,role,type,demographic,description,units,missing_values
0,buying,Feature,Categorical,None,buying price,None,no
1,maint,Feature,Categorical,None,price of the maintenance,None,no
2,doors,Feature,Categorical,None,number of doors,None,no
3,persons,Feature,Categorical,None,capacity in terms of persons to carry,None,no
4,lug_boot,Feature,Categorical,None,the size of luggage boot,None,no
5,safety,Feature,Categorical,None,estimated safety of the car,None,no
6,class,Target,Categorical,None,"evaulation level (unacceptable, acceptable, go...",None,no


El mismo creador del dataframe indica que no hay valores nulos e indica que todas las variables son categoricas.

In [22]:
X.shape

(1728, 6)

In [23]:
y.shape

(1728, 1)

Al haber mas de 50 registros no nulos, se puede usar estimador (https://scikit-learn.org/stable/machine_learning_map.html)



In [24]:
car_evaluation.variables

,name,role,type,demographic,description,units,missing_values
0,buying,Feature,Categorical,None,buying price,None,no
1,maint,Feature,Categorical,None,price of the maintenance,None,no
2,doors,Feature,Categorical,None,number of doors,None,no
3,persons,Feature,Categorical,None,capacity in terms of persons to carry,None,no
4,lug_boot,Feature,Categorical,None,the size of luggage boot,None,no
5,safety,Feature,Categorical,None,estimated safety of the car,None,no
6,class,Target,Categorical,None,"evaulation level (unacceptable, acceptable, go...",None,no


In [44]:
X.head()

,buying,maint,doors,persons,lug_boot,safety
0,vhigh,vhigh,2,2,small,low
1,vhigh,vhigh,2,2,small,med
2,vhigh,vhigh,2,2,small,high
3,vhigh,vhigh,2,2,med,low
4,vhigh,vhigh,2,2,med,med


In [53]:
X['buying'].unique()

['vhigh', 'high', 'med', 'low']
Categories (4, object): ['low' < 'med' < 'high' < 'vhigh']

las variables no vienen como ordinales

In [56]:
tamano = ["low", "med", "high", "vhigh"]

X["buying"] = pd.Categorical(
    X["buying"],
    categories = tamano,
    ordered = True
    )

In [58]:
X['maint'].unique()

['vhigh', 'high', 'med', 'low']
Categories (4, object): ['low' < 'med' < 'high' < 'vhigh']

In [57]:
X["maint"] = pd.Categorical(
    X["maint"],
    categories = tamano,
    ordered = True
    )

In [64]:
X['lug_boot'].unique()

['small', 'med', 'big']
Categories (3, object): ['small' < 'med' < 'big']

In [63]:
X["lug_boot"] = pd.Categorical(
    X["lug_boot"],
    categories = ['small', 'med', 'big'],
    ordered = True
    )

In [67]:
X['safety'].unique()

['low', 'med', 'high']
Categories (3, object): ['low' < 'med' < 'high']

In [66]:
X["safety"] = pd.Categorical(
    X["safety"],
    categories = ['low', 'med', 'high'],
    ordered = True
    )

In [72]:
y['class'].unique()

['unacc', 'acc', 'vgood', 'good']
Categories (4, object): ['unacc' < 'acc' < 'vgood' < 'good']

In [71]:
y['class'] = pd.Categorical(
    y['class'],
    categories = ['unacc', 'acc', 'vgood', 'good'],
    ordered = True
    )

Primer resumen de los datos:
* Hay 7 variables en el conjunto de datos. Todas las variables son de tipo de datos categóricos.
* Estos se dan por compra, mantenimiento, puertas, personas, lug_boot, seguridad y clase.
* La clase es la variable de destino o target.

In [80]:
X.columns

Index(['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety'], dtype='object')

In [81]:
# Exploremos un poco mas la variable target
print(X['buying'].value_counts(normalize=True))
print(X['maint'].value_counts(normalize=True))
print(X['doors'].value_counts(normalize=True))
print(X['persons'].value_counts(normalize=True))
print(X['lug_boot'].value_counts(normalize=True))
print(X['safety'].value_counts(normalize=True))

print(y['class'].value_counts(normalize=True))

buying
low      0.25
med      0.25
high     0.25
vhigh    0.25
Name: proportion, dtype: float64
maint
low      0.25
med      0.25
high     0.25
vhigh    0.25
Name: proportion, dtype: float64
doors
2        0.25
3        0.25
4        0.25
5more    0.25
Name: proportion, dtype: float64
persons
2       0.333333
4       0.333333
more    0.333333
Name: proportion, dtype: float64
lug_boot
small    0.333333
med      0.333333
big      0.333333
Name: proportion, dtype: float64
safety
low     0.333333
med     0.333333
high    0.333333
Name: proportion, dtype: float64
class
unacc    0.700231
acc      0.222222
good     0.039931
vgood    0.037616
Name: proportion, dtype: float64


los features son proporcionales lo cual es bueno

In [82]:
# calculando ratio de imbalance

ir = y.value_counts().max() / y.value_counts().min()

ir

18.615384615384617

Según ChatGPT hay desbalanceo al ser este numero mayor a 10

## Procesamiento de datos

In [ ]:
#Separamos en X e y


In [ ]:
#Importamos las librerias necesarias para la creacion del modelo


#30% para test y 70% para train


In [ ]:
#Veamos que obtuvimos


In [ ]:
#Veamos que tenemos. Por ejemplo, en X_train


## Entrenamiento de modelo de clasificación con árbol de decisión

In [ ]:
#Importante: todos nuestros tipos de datos son object, realizamos una transformacion


In [ ]:
#Verificamos la transformacion


In [ ]:
#Importar árbol de decisión

#Creacion del modelo


In [ ]:
#Entrenamiento


In [ ]:
#Calculo de las predicciones en Train y Test


## Evaluación de modelo de clasificación con árbol de decisión

In [ ]:
#Calculo de metricas


#Calculo el accuracy en Train


#Calculo el accuracy en Test


In [ ]:
#Verificamos el feature importances


## Entrenamiento de modelo de clasificación con random forest

In [ ]:
#Importar random forest


In [ ]:
#Calculo de las predicciones en Train y Test


## Evaluación de modelo de clasificación con random forest

In [ ]:
#Calculo de metricas


#Calculo el accuracy en Train


#Calculo el accuracy en Test


#Importante: podriamos reducir el numero de estimadores para disminuir el sobreajuste del modelo.

In [ ]:
# Visualizacion de las feature importantes


In [ ]:
#Grafico de barras


In [ ]:
# Matriz de confusion del RF


In [ ]:
#RF
